In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import librosa
from glob import glob
import random 
import numpy as np
fnames = glob("data/audio/birdsong-recognition/train_audio/**/*mp3")
random.shuffle(fnames)
tf.config.list_logical_devices()



[LogicalDevice(name='/device:CPU:0', device_type='CPU')]

In [2]:
def get_chunks(f):
    x,sr = librosa.load(f, sr=16000)
    N = 16000*3
    chunks = [ x[idx:idx+N] for idx in range(0, x.shape[0]-N, 200)]
    return chunks[:-1]
    
x = []
for f in fnames[:10]:
    try:
        x.append(np.vstack(get_chunks(f)))
    except:
        pass
x = np.vstack(x)
x.shape

Note: Illegal Audio-MPEG-Header 0x50455441 at offset 750153.
Note: Trying to resync...
Note: Hit end of (available) data during resync.


(35006, 48000)

In [ ]:
INPUT_DIM = 16000 * 3
LATENT_DIM = 32

def build_dense_autoencoder():
    # ----- Encoder -----
    inputs = layers.Input(shape=(INPUT_DIM,), name="input")

    x = layers.Dense(8192, activation="relu")(inputs)
    x = layers.Dense(2048, activation="relu")(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)

    latent = layers.Dense(
        LATENT_DIM,
        activation=None,
        name="embedding"
    )(x)

    # ----- Decoder -----
    x = layers.Dense(128, activation="relu")(latent)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dense(2048, activation="relu")(x)
    x = layers.Dense(8192, activation="relu")(x)

    outputs = layers.Dense(
        INPUT_DIM,
        activation=None,
        name="reconstruction"
    )(x)

    autoencoder = Model(inputs, outputs, name="dense_autoencoder")
    encoder = Model(inputs, latent, name="encoder")

    return autoencoder, encoder

model, embedding = build_dense_autoencoder()
model.compile(loss="mse", optimizer='adam', metrics=['mse'])
model.fit(x,x, epochs=50, validation_split=.1, batch_size=64)

Epoch 1/50


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

class EMAVectorQuantizer(layers.Layer):
    def __init__(
        self,
        num_embeddings=1024,
        embedding_dim=128,
        commitment_cost=0.25,
        decay=0.99,
        epsilon=1e-5,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        self.decay = decay
        self.epsilon = epsilon

    def build(self, input_shape):
        # Codebook
        self.embeddings = self.add_weight(
            shape=(self.num_embeddings, self.embedding_dim),
            initializer="uniform",
            trainable=False,
            name="embeddings"
        )

        # EMA buffers
        self.ema_cluster_size = self.add_weight(
            shape=(self.num_embeddings,),
            initializer="zeros",
            trainable=False,
            name="ema_cluster_size"
        )

        self.ema_embeddings = self.add_weight(
            shape=(self.num_embeddings, self.embedding_dim),
            initializer="uniform",
            trainable=False,
            name="ema_embeddings"
        )

    def call(self, inputs, training=None):
        """
        inputs: (..., embedding_dim)
        returns: quantized outputs, commitment loss
        """
        # Flatten input
        flat_inputs = tf.reshape(inputs, [-1, self.embedding_dim])

        # Compute distances
        distances = (
            tf.reduce_sum(flat_inputs**2, axis=1, keepdims=True)
            - 2 * tf.matmul(flat_inputs, self.embeddings, transpose_b=True)
            + tf.reduce_sum(self.embeddings**2, axis=1)
        )

        # Nearest codebook entry
        encoding_indices = tf.argmin(distances, axis=1)
        encodings = tf.one_hot(encoding_indices, self.num_embeddings)

        # Quantize
        quantized = tf.matmul(encodings, self.embeddings)
        quantized = tf.reshape(quantized, tf.shape(inputs))

        if training:
            # EMA updates
            updated_cluster_size = (
                self.decay * self.ema_cluster_size
                + (1 - self.decay) * tf.reduce_sum(encodings, axis=0)
            )

            dw = tf.matmul(encodings, flat_inputs, transpose_a=True)
            updated_ema_embeddings = (
                self.decay * self.ema_embeddings
                + (1 - self.decay) * dw
            )

            # Normalize cluster size
            n = tf.reduce_sum(updated_cluster_size)
            updated_cluster_size = (
                (updated_cluster_size + self.epsilon)
                / (n + self.num_embeddings * self.epsilon)
                * n
            )

            normalized_embeddings = (
                updated_ema_embeddings
                / tf.expand_dims(updated_cluster_size, -1)
            )

            self.ema_cluster_size.assign(updated_cluster_size)
            self.ema_embeddings.assign(updated_ema_embeddings)
            self.embeddings.assign(normalized_embeddings)

        # Commitment loss
        commitment_loss = self.commitment_cost * tf.reduce_mean(
            tf.square(tf.stop_gradient(quantized) - inputs)
        )

        # Straight-through estimator
        quantized = inputs + tf.stop_gradient(quantized - inputs)

        self.add_loss(commitment_loss)

        return quantized

    def get_code_indices(self, inputs):
        flat_inputs = tf.reshape(inputs, [-1, self.embedding_dim])
        distances = (
            tf.reduce_sum(flat_inputs**2, axis=1, keepdims=True)
            - 2 * tf.matmul(flat_inputs, self.embeddings, transpose_b=True)
            + tf.reduce_sum(self.embeddings**2, axis=1)
        )
        return tf.argmin(distances, axis=1)


In [ ]:
latent_dim = 128
vq = EMAVectorQuantizer(
    num_embeddings=1024,
    embedding_dim=latent_dim,
    commitment_cost=0.25,
    decay=0.99
)

# Example latent batch: (batch, time, latent_dim)
z = tf.random.normal([8, 100, latent_dim])

z_q = vq(z, training=True)


In [ ]:
token_ids = vq.get_code_indices(z)
token_ids = tf.reshape(token_ids, [8, 100])


In [ ]:
p = tf.reduce_mean(tf.one_hot(token_ids, 1024), axis=0)
perplexity = tf.exp(-tf.reduce_sum(p * tf.math.log(p + 1e-10)))
perplexity.numpy()